In [ ]:
import pandas as pd
import seaborn as sns
import os as os
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os

# -------------------------------
# 1. Raw data (from the existing regression df1 and df2)
# -------------------------------
metrics = [ "aMSE","aRMSE", "aCC", "aRRMSE", "EU_DIST"]

df1 = pd.DataFrame({
    "LGBM_MOR train": [0.164192,0.358481, 0.318395, 0.934759, 0.795362],
    "LGBM_MOR test":  [0.163768,0.357135, 0.334305, 0.929735, 0.797537],
    "LGBM_RC train":  [0.164248,0.358435, 0.319296, 0.934239, 0.792225],
    "LGBM_RC test":  [0.164915,0.358117, 0.332715, 0.930988, 0.797327],
    "MORF train":    [0.167686,0.361389, 0.313644, 0.938068, 0.815616],
    "MORF test":     [0.173035,0.366093, 0.273675, 0.947651, 0.830979],
}, index=metrics).round(4)

df2 = pd.DataFrame({
    "LGBM_MOR test":  [0.1514,0.3455, 0.3868, 0.9079, 0.7591],
    "LGBM_MOR train": [0.1027,0.2845, 0.7197, 0.7465, 0.6174],
    "RF_MOR test":    [0.1607,0.3537, 0.3608, 0.9207, 0.7972],
    "RF_MOR train":   [0.1397,0.3295, 0.5593, 0.8564, 0.7384],
    "LGBM_RC test":   [0.1758,0.3705, 0.2639, 0.9609, 0.8589],
    "LGBM_RC train":   [0.1608,0.3546, 0.3812, 0.9243, 0.8203],
    "RF_RC test":     [0.1804,0.3750, 0.2919, 0.9695, 0.7993],
    "RF_RC train":    [0.1669,0.3631, 0.3401, 0.9483, 0.7603],
    "MORF test":        [0.1629,0.3566, 0.3361, 0.9297, 0.8009],
    "MORF train":       [0.1483,0.3426, 0.4422, 0.9014, 0.7587],
}, index=metrics).round(4)
# -------------------------------
# 2. Tag data source (D1 / D2)
# -------------------------------
df1.columns = [f"{col} (D1)" for col in df1.columns]
df2.columns = [f"{col} (D2)" for col in df2.columns]

df_all = pd.concat([df1, df2], axis=1)

# -------------------------------
# 3. Split train/test
# -------------------------------
train_cols = [col for col in df_all.columns if "train" in col.lower()]
test_cols  = [col for col in df_all.columns if "test" in col.lower()]
df_train = df_all[train_cols]
df_test  = df_all[test_cols]

# -------------------------------
# 4. Normalization function (regression)
# -------------------------------
def normalize_regression(df):
    df_norm = df.copy()
    for row in df_norm.index:
        vals = df.loc[row].values.reshape(-1, 1)
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(vals).flatten()
        if row in ["aRMSE", "aRRMSE", "EU_DIST", "aMSE"]:  # lower is better
            df_norm.loc[row] = 1 - scaled
        else:  # CC: higher is better
            df_norm.loc[row] = scaled
    return df_norm

df_train_norm = normalize_regression(df_train)
df_test_norm  = normalize_regression(df_test)

# -------------------------------
# 5. Plotting function
# -------------------------------
def plot_horizontal_heatmap(df_train_raw, df_test_raw, df_train_norm, df_test_norm, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(df_train_raw.columns) * 0.45)), sharey=True)

    for ax, df_raw, df_norm, title in zip(
        axes, [df_train_raw, df_test_raw], [df_train_norm, df_test_norm], ["(a) Train Performance", "(b) Test Performance"]
    ):
        df_plot = df_raw.T
        df_annot = df_plot.applymap(lambda x: f"{x:.4f}")
        df_color = df_norm.T

         # Two-step approach: strip train/test, keep model name and (D1)/(D2)
        y_labels = df_plot.index.str.replace(' train ', ' ', regex=False)
        y_labels = y_labels.str.replace(' test ', ' ', regex=False)

        sns.heatmap(
            df_color,
            annot=df_annot,
            fmt="",
            cmap="RdBu",
            vmin=0,
            vmax=1,
            linewidths=0.5,
            ax=ax,
            cbar=False,
            annot_kws={"size": 10},   # heatmap annotation font size
            xticklabels=True,
             yticklabels=y_labels
        )
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Metric", fontsize=14)     # x-axis label font size
        ax.set_ylabel("Model", fontsize=14)      # y-axis label font size
        ax.tick_params(axis='x', labelsize=12,labelrotation=45)   # x-axis tick font size
        ax.tick_params(axis='y', labelsize=12)   # y-axis tick font size

    # colorbar
    cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=0, vmax=1))
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Bad", "Good"])
    cbar.ax.tick_params(labelsize=12)           # colorbar tick font size
    cbar.set_label("Performance Scale", fontsize=13)  # colorbar label font size

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()

# -------------------------------
# 6. Plot call
# -------------------------------
plot_horizontal_heatmap(df_train, df_test, df_train_norm, df_test_norm,
                        save_path="results/feature/regression_heatmap.png")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
import numpy as np
import os

# -------------------------------
# 1. Raw data (from the existing regression df1 and df2)
# -------------------------------
metrics = [ "aMSE","aRMSE", "aCC", "aRRMSE", "EU_DIST"]

df1 = pd.DataFrame({
    "LGBM_MOR train": [0.164192,0.358481, 0.318395, 0.934759, 0.795362],
    "LGBM_MOR test":  [0.163768,0.357135, 0.334305, 0.929735, 0.797537],
    "LGBM_RC train":  [0.164248,0.358435, 0.319296, 0.934239, 0.792225],
    "LGBM_RC test":  [0.164915,0.358117, 0.332715, 0.930988, 0.797327],
    "MORF train":    [0.167686,0.361389, 0.313644, 0.938068, 0.815616],
    "MORF test":     [0.173035,0.366093, 0.273675, 0.947651, 0.830979],
}, index=metrics).round(4)

df2 = pd.DataFrame({
    # MOR_LGBM
    "LGBM_MOR test":  [0.1533, 0.3477, 0.3623, 0.9175, 0.7597],
    "LGBM_MOR train": [0.1008, 0.2830, 0.7128, 0.7486, 0.6098],
     # MOR_RF
    "RF_MOR test":    [0.1615, 0.3544, 0.3418, 0.9261, 0.7979],
    "RF_MOR train":   [0.1399, 0.3301, 0.5527, 0.8601, 0.7398],
    # RC_LGBM
    "LGBM_RC test":   [0.1738, 0.3690, 0.2336, 0.9628, 0.8580],
    "LGBM_RC train":  [0.1569, 0.3519, 0.3829, 0.9225, 0.8161],
    # RC_RF
    "RF_RC test":     [0.1786, 0.3732, 0.2669, 0.9698, 0.7961],
    "RF_RC train":    [0.1649, 0.3609, 0.3597, 0.9437, 0.7584],

    # MORF
    "MORF test":      [0.1631, 0.3565, 0.3274, 0.9322, 0.8006],
    "MORF train":     [0.1485, 0.3428, 0.4370, 0.9033, 0.7598],

}, index=metrics).round(4)

print(df2)
# -------------------------------
# 2. Tag data source (D1 / D2)
# -------------------------------
df1.columns = [f"{col} (D1)" for col in df1.columns]
df2.columns = [f"{col} (D2)" for col in df2.columns]

df_all = pd.concat([df1, df2], axis=1)

# -------------------------------
# 3. Split train/test
# -------------------------------
train_cols = [col for col in df_all.columns if "train" in col.lower()]
test_cols  = [col for col in df_all.columns if "test" in col.lower()]
df_train = df_all[train_cols]
df_test  = df_all[test_cols]

# -------------------------------
# 4. Normalization function (regression)
# -------------------------------
def normalize_regression(df):
    df_norm = df.copy()
    for row in df_norm.index:
        vals = df.loc[row].values.reshape(-1, 1)
        scaler = MinMaxScaler()
        scaled = scaler.fit_transform(vals).flatten()
        if row in ["aRMSE", "aRRMSE", "EU_DIST", "aMSE"]:  # lower is better
            df_norm.loc[row] = 1 - scaled
        else:  # CC: higher is better
            df_norm.loc[row] = scaled
    return df_norm

df_train_norm = normalize_regression(df_train)
df_test_norm  = normalize_regression(df_test)

# -------------------------------
# 5. Plotting function
# -------------------------------
def plot_horizontal_heatmap(df_train_raw, df_test_raw, df_train_norm, df_test_norm, save_path=None):
    fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(df_train_raw.columns) * 0.45)), sharey=True)

    for ax, df_raw, df_norm, title in zip(
        axes, [df_train_raw, df_test_raw], [df_train_norm, df_test_norm], ["(a) Train Performance", "(b) Test Performance"]
    ):
        df_plot = df_raw.T
        df_annot = df_plot.applymap(lambda x: f"{x:.4f}")
        df_color = df_norm.T

         # Two-step approach: strip train/test, keep model name and (D1)/(D2)
        y_labels = df_plot.index.str.replace(' train ', ' ', regex=False)
        y_labels = y_labels.str.replace(' test ', ' ', regex=False)

        sns.heatmap(
            df_color,
            annot=df_annot,
            fmt="",
            cmap="RdBu",
            vmin=0,
            vmax=1,
            linewidths=0.5,
            ax=ax,
            cbar=False,
            annot_kws={"size": 10},   # heatmap annotation font size
            xticklabels=True,
             yticklabels=y_labels
        )
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("Metric", fontsize=14)     # x-axis label font size
        ax.set_ylabel("Model", fontsize=14)      # y-axis label font size
        ax.tick_params(axis='x', labelsize=12,labelrotation=45)   # x-axis tick font size
        ax.tick_params(axis='y', labelsize=12)   # y-axis tick font size

    # colorbar
    cbar_ax = fig.add_axes([0.92, 0.3, 0.015, 0.4])
    sm = plt.cm.ScalarMappable(cmap="RdBu", norm=plt.Normalize(vmin=0, vmax=1))
    cbar = fig.colorbar(sm, cax=cbar_ax)
    cbar.set_ticks([0, 1])
    cbar.set_ticklabels(["Bad", "Good"])
    cbar.ax.tick_params(labelsize=12)           # colorbar tick font size
    cbar.set_label("Performance Scale", fontsize=13)  # colorbar label font size

    plt.tight_layout(rect=[0, 0, 0.9, 1])
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300)
    plt.show()

# -------------------------------
# 6. Plot call
# -------------------------------
plot_horizontal_heatmap(df_train, df_test, df_train_norm, df_test_norm,
                        save_path="results/feature/regression_heatmap.png")